# Mystery & Sci-Fi — One-Click Colab Run

This is the straight-through runner for the first six-scene test. It does the setup, syncs the latest production code, checks your Colab Secret, prepares the episode, writes narration and timed captions, and creates the Google Flow manifest in one cell.

**Use:** Open in Google Colab and run the single code cell once. No separate setup cell, pasted patch, or Prepare button.

- Uses your existing Google Drive workspace and resumes files from the previous attempt.
- Uses the six-scene premise already selected.
- Keeps Gemini image API generation **off**; this does not spend separate image-generation API credits.
- Saves a persistent log in the project folder, including the last output if a run fails.
- Google Flow video generation is still a manual step after preparation; this notebook will not spend Flow credits automatically.


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, html as html_lib
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Markdown, HTML

# Fixed for the first six-scene test; edit these values only when making a different episode.
TOPIC = "An abandoned coastal radio station receives a warning about an event that has not happened yet."
SCENE_COUNT = 6
VOICE = "en-US-ChristopherNeural"
WHISPER_MODEL = "base.en"
WHISPER_DEVICE = "cpu"
WHISPER_COMPUTE_TYPE = "int8"
WHISPER_DEVICE_PREFERENCE = "auto"
PROJECT_ID = "20261120_123020_mystery_scifi_An_abandoned_coastal_radio_station_recei"
RESUME_PROJECT_ID = PROJECT_ID
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
ROOT = Path("/content/gta6-autoclipper")
LOG_PATH = None

try:
    # Validate the key before Drive mounting, cloning, package installation, or API calls.
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    api_key = str(api_key or os.environ.get("GEMINI_API_KEY", "")).strip()
    if len(api_key) < 10:
        raise RuntimeError(
            "GEMINI_API_KEY is missing or inaccessible. Add it under Colab Secrets "
            "with notebook access enabled, or provide it in the environment. "
            "The run was stopped before production work or API calls."
        )

    if not Path("/content/drive/MyDrive").is_dir():
        print("Connecting Google Drive…", flush=True)
        drive.mount("/content/drive")

    WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
    PROJECT_DIR = WORKSPACE / "projects" / RESUME_PROJECT_ID
    OUTPUT_DIR = PROJECT_DIR / "mystery_scifi"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    LOG_PATH = OUTPUT_DIR / "mystery_run.log"

    def note(message):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {message}"
        print(line, flush=True)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write(line + "\n")
            log.flush()

    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n\n===== One-click six-scene run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n")

    note("Persistent log: " + str(LOG_PATH))
    note("Project output: " + str(OUTPUT_DIR))

    def run_logged(label, command, env=None):
        note("START: " + label)
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("COMMAND: " + " ".join(str(part) for part in command) + "\n")
            log.flush()
            process = subprocess.Popen(
                [str(part) for part in command],
                cwd=str(ROOT) if ROOT.is_dir() else None,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                errors="replace",
            )
            log.write("PROCESS_ID: " + str(process.pid) + "\n")
            log.flush()
            print("Process started: " + str(process.pid), flush=True)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            log.write("EXIT_CODE: " + str(return_code) + "\n")
            log.flush()
        if return_code != 0:
            raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
        note("DONE: " + label)

    def topic_tokens(value):
        import re
        ignored = {
            "a", "an", "and", "are", "as", "at", "by", "for", "from", "in",
            "into", "is", "it", "of", "on", "or", "the", "that", "this", "to",
            "was", "were", "with",
        }
        return {
            token for token in re.findall(r"[a-z0-9]+", str(value or "").casefold())
            if len(token) > 1 and token not in ignored
        }

    def topics_differ_significantly(saved_topic, requested_topic):
        import re
        saved_normalized = " ".join(re.findall(r"[a-z0-9]+", str(saved_topic or "").casefold()))
        requested_normalized = " ".join(re.findall(r"[a-z0-9]+", str(requested_topic or "").casefold()))
        if not saved_normalized or not requested_normalized or saved_normalized == requested_normalized:
            return False
        saved_tokens = topic_tokens(saved_topic)
        requested_tokens = topic_tokens(requested_topic)
        if not saved_tokens or not requested_tokens:
            return True
        similarity = len(saved_tokens & requested_tokens) / max(1, len(saved_tokens | requested_tokens))
        return similarity < 0.30

    if (ROOT / ".git").is_dir():
        run_logged("Sync latest GitHub production code", ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", "main"])
        run_logged("Select the latest tested main commit", ["git", "-C", str(ROOT), "reset", "--hard", "FETCH_HEAD"])
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged("Download production code", ["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])

    # Resume takes priority over the newly entered TOPIC when the saved project is
    # materially different. Warn first and preserve that plan; never delete it by count.
    plan_path = OUTPUT_DIR / "episode.json"
    saved_plan_topic = None
    resume_topic_override = False
    if plan_path.is_file():
        try:
            previous = json.loads(plan_path.read_text(encoding="utf-8"))
            saved_plan_topic = str(previous.get("topic", "")).strip() or None
            previous_count = len(previous.get("scenes", []))
        except Exception as parse_error:
            previous_count = -1
            note(f"WARNING: Existing episode.json could not be read ({type(parse_error).__name__}: {parse_error}); it will be backed up before a new plan is generated.")
        if saved_plan_topic and topics_differ_significantly(saved_plan_topic, TOPIC):
            resume_topic_override = True
            note(
                f"WARNING: RESUME_PROJECT_ID='{RESUME_PROJECT_ID}' is overriding TOPIC. "
                f"Saved project plan topic: {saved_plan_topic!r}. Current TOPIC: {TOPIC!r}. "
                "The existing project's plan will be reused unchanged."
            )
        elif previous_count != SCENE_COUNT:
            backup = OUTPUT_DIR / ("episode.backup.before_scene_count_change_" + time.strftime("%Y%m%d_%H%M%S") + ".json")
            shutil.copy2(plan_path, backup)
            plan_path.unlink()
            saved_plan_topic = None
            note(f"Existing plan had {previous_count} scenes; backed it up to {backup} before generating the requested {SCENE_COUNT}-scene plan.")
        else:
            note("Found a saved plan with the requested topic and scene count; it will be reused.")

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "project_state.json").write_text(
        json.dumps({
            "project_name": RESUME_PROJECT_ID,
            "mode": "Mystery & Sci-Fi Automation",
            "story_title": (saved_plan_topic or TOPIC)[:160],
            "mystery_output": str(OUTPUT_DIR),
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "setup": "one-click-colab",
        }, indent=2),
        encoding="utf-8",
    )

    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run_logged(
            "Install FFmpeg",
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core"],
        )
    else:
        note("FFmpeg and ffprobe are available.")

    note("Install/check production dependencies; image-generation API stays disabled.")
    run_logged("Install Python dependencies", [
        sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
        "google-genai>=2.12.1,<3",
        "pydantic>=2.7,<3",
        "python-dotenv>=1.0,<2",
        "edge-tts>=7.0,<9",
        "av==17.1.0",
        "ctranslate2==4.9.1",
        "faster-whisper==1.2.1",
        "Pillow>=10,<13",
    ])

    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "GEMINI_API_KEY": api_key,
        "ALLOW_PAID_IMAGE_API": "false",
        "MYSTERY_WHISPER_DEVICE": WHISPER_DEVICE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "RESUME_PROJECT_ID": PROJECT_ID,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(OUTPUT_DIR / "no_music_selected.mp3"),
    })
    command = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", TOPIC,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", WHISPER_DEVICE,
        "--max-scenes-per-flow-run", "10",
    ]
    # Deliberately omit --allow-paid-images.
    run_logged("Prepare six-scene episode, narration, captions, and Flow manifest", command, env=env)

    required_files = [
        OUTPUT_DIR / "episode.json",
        OUTPUT_DIR / "episode_brief.md",
        OUTPUT_DIR / "assets_generation_plan.md",
        OUTPUT_DIR / "audio" / "narration_manifest.json",
        OUTPUT_DIR / "captions" / "subtitles.srt",
        OUTPUT_DIR / "flow_upload_manifest.json",
        OUTPUT_DIR / "flow_upload_manifest.txt",
        OUTPUT_DIR / "flow_master_prompt.md",
    ]
    missing = [str(path) for path in required_files if not path.is_file()]
    if missing:
        raise RuntimeError("The process exited successfully but required output files are missing: " + ", ".join(missing))

    episode = json.loads((OUTPUT_DIR / "episode.json").read_text(encoding="utf-8"))
    if len(episode.get("scenes", [])) != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {len(episode.get('scenes', []))}.")

    note("SUCCESS: six-scene episode plan, narration, timed captions, and Flow manifest verified.")
    display(Markdown(
        "## Six-scene preparation complete\n\n"
        f"**Episode:** {episode.get('title', 'Untitled')}\n\n"
        f"**Scenes:** {len(episode['scenes'])}\n\n"
        f"**Project folder:** {OUTPUT_DIR}\n\n"
        f"**Flow manifest:** {OUTPUT_DIR / 'flow_upload_manifest.txt'}\n\n"
        f"**Master prompt:** {OUTPUT_DIR / 'flow_master_prompt.md'}\n\n"
        f"**Narration:** {OUTPUT_DIR / 'audio'}\n\n"
        f"**Captions:** {OUTPUT_DIR / 'captions' / 'subtitles.srt'}\n\n"
        f"**Persistent run log:** {LOG_PATH}\n\n"
        "Image API calls were disabled. Video creation in Google Flow remains a manual stage; the manifest "
        "lists the six clips in order."
    ))

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    try:
        with LOG_PATH.open("a", encoding="utf-8") as log:
            log.write("\n" + error + "\n")
            log.write(traceback.format_exc() + "\n")
            log.flush()
        tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
        print("\nLast log lines:", flush=True)
        print("\n".join(tail), flush=True)
        print("\nFull persistent log: " + str(LOG_PATH), flush=True)
    except Exception:
        print(traceback.format_exc(), flush=True)
